# Fan et al. 2023: neural to characters across days (CTC)

Decodes the **go-period threshold crossings of one trial** into the **character sequence** of the sentence T5 attempted to write, with a CTC loss. The target comes from `Sentence.text` through the frozen `CharacterEncoder` charset (blank, space, apostrophe, `a`-`z`). The metric is the character error rate (CER).

The question this dataset asks is whether a decoder trained on earlier days still works on later ones. So the split is by day:

- **train:** every valid trial of the Dryad days except one day, seed and online blocks alike,
- **validation:** that day, 2022-12-15 (both block kinds), used to pick the epoch,
- **test:** the valid FALCON held-out days, 2023-04-17, 2023-05-31 and 2023-06-28 (3 trials each, 9 in all), scored once.

Nine trials make a noisy test number, so the per-day CER is printed beside the pooled one. The three days flagged `valid=False` stay out through the Study query. The features drift between days, so each day is normalized with its own statistics. Trials last up to 94 s, so the input is read at 12.5 Hz to keep training short.

The setup cell builds the Study and calls `download()`, which fetches the Dryad deposit and the published dandiset on the first run and returns immediately afterwards.

In [1]:
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

from neuralshed.dataloader import PadSpec, RaggedCollator, RaggedSegmenter
from neuralshed.datasets.fan_2023 import study
from neuralshed.datasets.utils import dataset_dir
from neuralshed.extractors import CharacterEncoder, NeuralFeatureExtractor
from neuralshed.text import CharVocab, ctc_min_input_length

pd.set_option("display.max_columns", None)
torch.manual_seed(0)

# reads ./.env into os.environ
load_dotenv()

root = dataset_dir("fan_2023") / "data"
s = study.Fan2023Handwriting(path=root)
s.download()

# input rate, a quarter of the native 50 Hz
FS = 12.5
VAL_DATE = "2022-12-15"
EPOCHS = 30

# 29 classes: blank + space + apostrophe + a-z
VOCAB = CharVocab.canonical()

# extractor disk cache
CACHE = dict(folder=dataset_dir("fan_2023") / "cache", keep_in_ram=False)


def conv_len(n):
    """Output length of one conv with kernel 5, stride 2, padding 2."""
    return (n + 2 * 2 - 5) // 2 + 1


def frame_mask(lengths, n_frames):
    """(B, 1, n_frames) mask that is 1 on each trial's valid frames."""
    return (
        (torch.arange(n_frames)[None, :] < lengths[:, None])
        .unsqueeze(1)
        .to(torch.float32)
    )


def levenshtein(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev = d[0]
        d[0] = i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[-1]


def greedy(logp, n):
    """(T', C) log-probs over n valid frames -> collapsed character ids."""
    out = []
    prev = -1
    for v in logp[:n].argmax(-1).tolist():
        if v != prev and v != VOCAB.BLANK:
            out.append(v)
        prev = v
    return out


print(VOCAB)

2026-10-14 16:43:45 - INFO - neuralset.events.study:463 - Success: Study downloaded to /Volumes/seagate_0/tether/datasets/fan_2023/data/Fan2023Handwriting.


2026-10-14 16:43:45 - INFO - neuralset.events.study:85 - Permissions set for /Volumes/seagate_0/tether/datasets/fan_2023/data/Fan2023Handwriting (skipped 0 non-owned items).


CharVocab(version='v1', n_classes=29)


## 1. Valid days, split by day

The split is set from the day: FALCON's held-out days are the test set, the validation day is held out of training, and every other day trains. Every trial of a day lands on one side.

In [2]:
events = study.Fan2023Handwriting(path=root, query="valid").run()
events["split"] = "train"
events.loc[events.date == VAL_DATE, "split"] = "val"
events.loc[events.original_split == "held-out-calib", "split"] = "test"

sentences = events[events.type == "Sentence"]
print("trials:", len(sentences), "| days:", sentences.date.nunique())
display(
    sentences.groupby(["split", "block_kind"]).agg(
        days=("date", "nunique"), trials=("trial_id", "size")
    )
)
print(
    "days on more than one side:",
    int((sentences.groupby("date").split.nunique() > 1).sum()),
)

trials: 1264 | days: 23


days  trials
split block_kind                    
test  recalibration        3       9
train no_recalibration     8     160
      recalibration        8     317
      seed                11     739
val   no_recalibration     1      19
      recalibration        1      20

days on more than one side: 0


## 2. One segment per trial

- `neural`: the trial's `SpikeCounts` read at 12.5 Hz in `Hz`, `(192, T)`. The extractor averages groups of four 20 ms bins, and converting to a rate keeps the values independent of the bin width.
- `target`: the character ids of the `Sentence` through the frozen vocabulary.

Each segment covers the trial's go period, which is all the deposits release. A CTC target needs at least `len(ids) + adjacent repeats` emission frames. The model emits at 6.25 Hz, several times T5's writing speed of about 1 character per second. The cell still checks and reports any trial that does not fit.

In [3]:
segmenter = RaggedSegmenter(
    extractors={
        "neural": NeuralFeatureExtractor(
            target_unit="Hz", event_types="SpikeCounts", frequency=FS, infra=CACHE
        ),
        "target": CharacterEncoder(aggregation="trigger"),
    },
    trigger_query="type == 'Sentence'",
    # each segment inherits its sentence duration
    duration=None,
    metadata_columns=("split", "text", "timeline", "date"),
)
dataset = segmenter.apply(events)
dataset.prepare()

sample = dataset[0]
print(
    "neural:",
    tuple(sample.data["neural"].shape),
    "| target:",
    tuple(sample.data["target"].shape),
)
print("text  :", repr(sample.metadata["text"]))
print("ids   :", repr(VOCAB.decode(sample.data["target"].tolist())))

keep = []
for i in range(len(dataset)):
    item = dataset[i]
    frames = conv_len(item.data["neural"].shape[-1])
    keep.append(frames >= ctc_min_input_length(item.data["target"].tolist()))
dataset = dataset.select(np.array(keep))
print(dataset)
print("dropped for too few emission frames:", len(keep) - sum(keep))

2026-10-14 16:43:54 - INFO - neuralset.dataloader:222 - Preparing extractor: NeuralFeatureExtractor


2026-10-14 16:43:54 - INFO - neuralset.dataloader:222 - Preparing extractor: CharacterEncoder


neural: (192, 672) | target: (41,)
text  : 'there just might be too many people there.'
ids   : 'there just might be too many people there'


RaggedSegmentDataset(1264 segments, extractors: neural, target, metadata: split, text, timeline, date)
dropped for too few emission frames: 0


## 3. Per-day normalization

Each day's per-channel mean and standard deviation come from all frames of that day's trials. That uses the day's neural data and no labels, so a test day is normalized by itself, as an online system recalibrating on a new day would be. Fan et al. used rolling statistics within each block, which this approximates at the day level.

In [4]:
collate = RaggedCollator(
    fields={
        "neural": PadSpec(axis=-1, pad_value=0),
        "target": PadSpec(axis=0, pad_value=0),
    }
)


def loader(split, shuffle=False):
    subset = dataset.select(dataset.metadata.split.eq(split).to_numpy())
    return DataLoader(subset, batch_size=16, shuffle=shuffle, collate_fn=collate)


train_loader = loader("train", shuffle=True)
val_loader = loader("val")
test_loader = loader("test")
print({split: len(loader(split).dataset) for split in ("train", "val", "test")})

total = {}
total_sq = {}
n_frames = {}
for b in DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=collate):
    mask = b.valid_mask("neural").unsqueeze(1)
    x = b.data["neural"] * mask
    for k, date in enumerate(b.metadata["date"]):
        total[date] = total.get(date, torch.zeros(192)) + x[k].sum(-1)
        total_sq[date] = total_sq.get(date, torch.zeros(192)) + (x[k] ** 2).sum(-1)
        n_frames[date] = n_frames.get(date, 0) + int(mask[k].sum())

day_mu = {}
day_sd = {}
for date in total:
    mu = total[date] / n_frames[date]
    day_mu[date] = mu
    day_sd[date] = (total_sq[date] / n_frames[date] - mu**2).clamp_min(0).sqrt() + 1e-3
print(
    f"{len(day_mu)} days, {min(n_frames.values())}-{max(n_frames.values())} frames per day"
)


def normalized(batch):
    """(B, 192, T) input with its day's statistics removed, padded frames kept at zero."""
    mu = torch.stack([day_mu[d] for d in batch.metadata["date"]]).unsqueeze(-1)
    sd = torch.stack([day_sd[d] for d in batch.metadata["date"]]).unsqueeze(-1)
    x = (batch.data["neural"] - mu) / sd
    return x * batch.valid_mask("neural").unsqueeze(1)

{'train': 1216, 'val': 39, 'test': 9}


23 days, 993-61704 frames per day


## 4. A compact CTC decoder

One strided temporal conv (12.5 Hz to 6.25 Hz), a 2-layer bidirectional GRU and a per-frame linear head over the **29 classes**, read as CTC emissions. The GRU runs on packed sequences, so a trial's emissions do not depend on the padding of its batch.

In [5]:
class CTCDecoder(nn.Module):
    def __init__(self, c_in=192, n_classes=VOCAB.n_classes, hidden=128, p=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(c_in, 192, 5, 2, 2), nn.ReLU(), nn.Dropout(p)
        )
        self.gru = nn.GRU(
            192, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=p
        )
        self.head = nn.Linear(2 * hidden, n_classes)

    def forward(self, x, lengths):
        # x: (B, 192, T), lengths: (B,) valid frames
        h = self.conv(x)
        n_valid = conv_len(lengths)
        # zero the frames past each trial's valid length, so the GRU packs only real frames
        h = h * frame_mask(n_valid, h.shape[-1])
        packed = pack_padded_sequence(
            h.transpose(1, 2), n_valid.cpu(), batch_first=True, enforce_sorted=False
        )
        packed, _ = self.gru(packed)
        h, _ = pad_packed_sequence(packed, batch_first=True, total_length=h.shape[-1])
        return self.head(h).log_softmax(-1), n_valid


model = CTCDecoder()
print(model)
print(sum(p.numel() for p in model.parameters()), "parameters")

CTCDecoder(
  (conv): Sequential(
    (0): Conv1d(192, 192, kernel_size=(5,), stride=(2,), padding=(2,))
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (gru): GRU(192, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (head): Linear(in_features=256, out_features=29, bias=True)
)
735709 parameters


## 5. Train and pick the epoch on the validation day

CTC sits on an all-blank plateau for the first epochs, and a one-cycle schedule keeps the learning rate high through it. The weights with the lowest validation CER are kept. The test days are not looked at until the next section.

In [6]:
def cer(loader):
    """Character error rate: Levenshtein edits over reference characters, summed over trials."""
    model.eval()
    err = 0
    tot = 0
    with torch.no_grad():
        for b in loader:
            logp, n_valid = model(normalized(b), b.lengths["neural"])
            for k in range(len(n_valid)):
                ref = b.data["target"][k, : b.lengths["target"][k]].tolist()
                err += levenshtein(greedy(logp[k], int(n_valid[k])), ref)
                tot += len(ref)
    return err / tot


opt = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=3e-3, total_steps=EPOCHS * len(train_loader)
)
ctc = nn.CTCLoss(blank=VOCAB.BLANK, zero_infinity=True)

best_cer = float("inf")
best_epoch = 0
best_state = None
for ep in range(EPOCHS):
    model.train()
    run = 0.0
    nb = 0
    for b in train_loader:
        logp, n_valid = model(normalized(b), b.lengths["neural"])
        loss = ctc(
            logp.permute(1, 0, 2), b.data["target"], n_valid, b.lengths["target"]
        )
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        run += loss.item()
        nb += 1
    val_cer = cer(val_loader)
    if val_cer < best_cer:
        best_cer = val_cer
        best_epoch = ep + 1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if (ep + 1) % 5 == 0:
        print(f"epoch {ep + 1:2d}  train CTC {run / nb:5.3f}  val CER {val_cer:.3f}")

model.load_state_dict(best_state)
print(f"kept epoch {best_epoch}, val CER {best_cer:.3f}")

epoch  5  train CTC 1.458  val CER 0.289


epoch 10  train CTC 0.229  val CER 0.138


epoch 15  train CTC 0.132  val CER 0.119


epoch 20  train CTC 0.068  val CER 0.101


epoch 25  train CTC 0.040  val CER 0.090


epoch 30  train CTC 0.034  val CER 0.087
kept epoch 27, val CER 0.085


## 6. Evaluate on the held-out days

The reference point is the empty prediction (CER 1.000), where every character is a deletion. This CER is a tutorial number on released trials. It is not comparable with FALCON's benchmark, which scores word and character error rates on the withheld trials of the held-out days.

In [7]:
print(f"test CER (9 trials) : {cer(test_loader):.3f}")
print(f"val CER             : {cer(val_loader):.3f}")
print(f"train CER           : {cer(train_loader):.3f}")
print("baseline (empty)    : 1.000")

test_ds = test_loader.dataset
for date in sorted(set(test_ds.metadata.date)):
    day = test_ds.select(test_ds.metadata.date.eq(date).to_numpy())
    day_loader = DataLoader(day, batch_size=16, shuffle=False, collate_fn=collate)
    print(f"test CER {date}: {cer(day_loader):.3f}  ({len(day)} trials)")

test CER (9 trials) : 0.252


val CER             : 0.085


train CER           : 0.002
baseline (empty)    : 1.000
test CER 2023-04-17: 0.123  (3 trials)
test CER 2023-05-31: 0.271  (3 trials)
test CER 2023-06-28: 0.324  (3 trials)


## 7. A decoded test trial against its target

Greedy CTC decoding without a language model.

In [8]:
# shortest test target
pick = min(
    (test_ds[i] for i in range(len(test_ds))), key=lambda t: len(t.data["target"])
)
b = collate([pick])
model.eval()
with torch.no_grad():
    logp, n_valid = model(normalized(b), b.lengths["neural"])
pred = greedy(logp[0], int(n_valid[0]))
ref = pick.data["target"].tolist()
print("sentence :", repr(pick.metadata["text"]))
print("target   :", repr(VOCAB.decode(ref)))
print("decoded  :", repr(VOCAB.decode(pred)))
print(f"CER      : {levenshtein(pred, ref) / len(ref):.3f}")

sentence : 'go to sleep, he said.'
target   : 'go to sleep he said'
decoded  : 'o fo slep he said'
CER      : 0.158


## Notes

- **Protocol.** Valid days only, train on every Dryad trial of the other days, pick the epoch on 2022-12-15, test once on the valid FALCON held-out days. Each day normalized with its own frames, 12.5 Hz input, a length-aware CTC model, greedy decoding.
- **The test set is small.** FALCON releases three trials per held-out day, and two of the five held-out days are flagged, so the test number rests on 9 sentences. The online decoder's own CER on the Dryad days is in `01_explore_source_data.ipynb`, for scale.
- **The authors' recipe**, for a decoder meant to be compared with the paper: rolling z-scoring within each block, Gaussian smoothing, a day-specific input layer in front of a 2-layer 512-unit GRU, and a 3-gram language model with GPT-2 rescoring (Fan et al. 2023 and FALCON appendix A.4.6).
- **What would move the number.** A day-specific input layer, more units, a language model over the CTC outputs, and a charset with the comma, period and question mark.